# Comparacion de modelos de clasificacion HistGradientBoosting, LightGBM, CatBoost, XGBoost

In [33]:
# Standard libraries
import os
import warnings

In [34]:

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [35]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, confusion_matrix, classification_report
)
from sklearn.calibration import CalibratedClassifierCV

In [37]:

from typing import Dict, Any, List, Optional, Tuple
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier

# --- add near imports ---
import os, json, re
from pathlib import Path
from datetime import datetime
import joblib

import os, json
from pathlib import Path
from datetime import datetime
import joblib
import re



In [38]:
# Statistical distributions
from scipy.stats import loguniform, randint, uniform

# Typing utilities
from typing import Dict, Any, List, Optional, Tuple

In [39]:
sns.set(style="ticks", context="notebook", palette="deep")
pd.set_option('display.max_columns', None)
palette = {'Bad':'#b2182b','Poor':'#d6604d','Moderate':'#f1a340','Good':'#5aae61','High':'#1b7837'}

In [40]:
path = "../../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
df1 = pd.read_parquet(os.path.join(path, "taxones_pressure_train.parquet"))
df2 = pd.read_parquet(os.path.join(path, "taxones_pressure_predict.parquet"))
df = pd.concat([df1, df2], ignore_index=True)

In [41]:
# change unassessed values to NaN
df = df.replace("Unassessed", np.nan)
df = df.replace("None", np.nan)

# index SamplingOperations_code
df = df.set_index('SamplingOperations_code')
 
# DROP HERlvl2Code	Altitude Longitude_Lambert93	Latitude_Lambert93	Watershed	CodeDepartement	HERlvl1Code
df = df.drop(columns=['HERlvl2Code', 'HERlvl2Name', 'Altitude','Longitude_Lambert93','Latitude_Lambert93','Watershed','CodeDepartement', 'Date_SamplingOperation'])
df

CodeSite_SamplingOperations  \
SamplingOperations_code                               
S02000008_20170703                        S02000008   
S02000008_20200708                        S02000008   
S02000010_20070906                        S02000010   
S02000010_20090721                        S02000010   
S02000010_20110723                        S02000010   
...                                             ...   
S06940940_20100708                        S06940940   
S06940940_20230623                        S06940940   
S06960950_20160629                        S06960950   
S06960950_20180719                        S06960950   
S06970900_20150601                        S06970900   

                         TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000008_20170703                                    405      NaN      NaN   
S02000008_20200708                                    400      NaN      NaN   
S02000010_20070906                                    400      NaN      NaN   
S02000010_20090721                                    400      NaN      NaN   
S02000010_20110723                                    398      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000008_20170703           NaN        NaN      NaN      NaN      NaN   
S02000008_20200708           NaN        NaN      NaN      NaN      NaN   
S02000010_20070906           NaN        NaN      NaN      NaN      NaN   
S02000010_20090721           NaN        NaN      NaN      NaN      NaN   
S02000010_20110723           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S0

In [10]:
# keep the numerical columns only
df = df.select_dtypes(include=[np.number, 'category'])
df

TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000008_20170703                                    405      NaN      NaN   
S02000008_20200708                                    400      NaN      NaN   
S02000010_20070906                                    400      NaN      NaN   
S02000010_20090721                                    400      NaN      NaN   
S02000010_20110723                                    398      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000008_20170703           NaN        NaN      NaN      NaN      NaN   
S02000008_20200708           NaN        NaN      NaN      NaN      NaN   
S02000010_20070906           NaN        NaN      NaN      NaN      NaN   
S02000010_20090721           NaN        NaN      NaN      NaN      NaN   
S02000010_20110723           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN  2.283105      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN       NaN      NaN      NaN      NaN   

                         Achca04  Achch01  Achcl01  Achco01  Achco02  Achco03  \
SamplingOperations_code                              

In [11]:
import cleandf


df1 = cleandf.IWANTMYXCLEAN(df, thresh_high_missing=.99)


Dropped exact duplicate columns: ['Cocps01', 'Dipde01', 'Dippa01', 'Encsu01', 'Encsu02', 'Ethpu01', 'Grama01', 'Navpa06', 'Navth01', 'Navtr01', 'Nitth01', 'Parhe01', 'Plata01', 'Semro01', 'Synaf01', 'Thani01']
Dropped columns with >99% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achal01', 'Acham01', 'Achan01', 'Achat01', 'Achat03', 'Achba01', 'Achbi01', 'Achbi02', 'Achbr01', 'Achca01', 'Achca03', 'Achca04', 'Achch01', 'Achcl01', 'Achco01', 'Achco03', 'Achco04', 'Achcr01', 'Achcy01', 'Achde01', 'Achdi01', 'Achdi02', 'Achel01', 'Achen01', 'Achex02', 'Achex03', 'Achfl01', 'Achfr01', 'Achgr01', 'Achgr03', 'Achha01', 'Achhe01', 'Achhi01', 'Achho01', 'Achho02', 'Achim01', 'Achim02', 'Achim03', 'Achin01', 'Achin02', 'Achja01', 'Achja02', 'Achjo01', 'Achkr01', 'Achkr03', 'Achkr04', 'Achku01', 'Achla01', 'Achla05', 'Achle01', 'Achle02', 'Achli01', 'Achli02', 'Achli04', 'Achlo01', 'Achlu01', 'Achlu02', 'Achma01', 'Achmi01', 'Achmo01', 'Achmo02', 'Achna01', 'Achna02', 'Achne01', 'Achni01', 'Achno

In [12]:
df1

,TotalAbundance_SamplingOperation,Achaf02,Achat02,Achca02,Achco02,Achda01,Achda02,Achde02,Achde03,Achdr01,Acheu01,Achex01,Achfu01,Achge01,Achgr02,Achho03,Achhu01,Achko01,Achkr02,Achla02,Achla03,Achla04,Achla06,Achli03,Achmi02,Achmi03,Achob01,Achpy01,Achri01,Achst03,Achsu01,Achsu03,Achth01,Adlbr02,Adlmi01,Ampat01,Ampin01,Ampin02,Ampma01,Ampmo01,Ampov01,Amppe01,Amppe02,Ampve01,Astfo01,Aulam01,Aulgr01,Bacpa01,Calae01,Calba01,Calfo01,Cocdi01,Coceu01,Cochu01,Cocli01,Cocne03,Cocpa01,Cocpe01,Cocpl01,Cocps02,Conwe01,Crasu01,Cycat01,Cycdi01,Cycdu01,Cycme01,Cycme02,Cycmi01,Cycps01,Cycst01,Cycwo01,Cymaf01,Cymaf02,Cymco01,Cymex01,Cymex02,Cymmi01,Cymne03,Cymsi01,Cymso01,Cymtu01,Cymtu03,Cymve01,Denta01,Dente01,Diaco01,Diaeh01,Diame01,Diamo01,Diavu01,Dipma01,Dippa02,Dipse01,Encca01,Encmi03,Encmi04,Encne01,Encsi02,Encsu03,Eolco01,Eolrh01,Eunbi05,Eunbo03,Eunim01,Eunin01,Eunte01,Exiva01,Falpy01,Frabr01,Fraca02,Fraca04,Fraco01,Fraex01,Fragr01,Frala02,Frami03,Frane04,Frapa01,Frapa02,Frape01,Frare01,Fraru01,Frave01,Fravi02,Fruco01,Fruve01,Fruvu01,Geiac01,Gomac03,Goman02,Goman05,Gombo02,Gomca02,Gomcu02,Gomel01,Gomex01,Gomin06,Gomit01,Gomla05,Gomli05,Gommi01,Gommi02,Gommi03,Gommi05,Gomol04,Gompa06,Gompu02,Gomrh01,Gomte02,Gomtr02,Gyrac01,Gyrat01,Halmo01,Hanam01,Hanar01,Himex01,Himmi01,Hipca02,Hombu01,Humco01,Karcl01,Karpl01,Lutgo01,Mayin01,Melgr01,Melpu01,Melte01,Melva01,Merci01,Merco01,Micpo01,Monpr01,Nantr01,Navac02,Navag03,Navam05,Navan02,Navan05,Navar02,Navar04,Navat02,Navca07,Navca08,Navca10,Navca11,Navci01,Navco05,Navco06,Navcr04,Navcr05,Navcr09,Navcr10,Navde03,Navdi04,Naver01,Navev01,Navex02,Navge02,Navgr01,Navhe04,Navjo01,Navla04,Navle02,Navme04,Navmi09,Navmo05,Navmo08,Navmu02,Navmu06,Navno02,Navob04,Navob06,Navoc01,Navpe05,Navpe06,Navpr03,Navps01,Navps02,Navra01,Navre04,Navrh01,Navrh02,Navsa03,Navsc04,Navsi04,Navsu03,Navsu07,Navsu08,Navsu09,Navsy01,Navte01,Navtr05,Navtr07,Navut01,Navva03,Navve03,Navvi01,Navvi02,Nitac04,Nitac05,Nitad01,Nitag02,Nitam01,Nitan02,Nitar01,Nitbe01,Nitca03,Nitco07,Nitde03,Nitdi04,Nitdr01,Nitfi02,Nitfo01,Nitfr01,Nitfr02,Nitgr02,Nitha02,Nithe01,Nithu01,Nitin02,Nitin04,Nitla03,Nitli03,Nitlo01,Nitme01,Nitmi01,Nitpa01,Nitpa02,Nitpe03,Nitpu04,Nitre01,Nitre02,Nitsi02,Nitso01,Nitso02,Nitso04,Nitsu01,Nitsu05,Nitsu09,Nitte02,Nupla01,Panco02,Panoc01,Parpr01,Plaap01,Plaba01,Placl01,Pladu01,Plala02,Plami02,Plaob01,Plaro03,Plaro04,Pleku01,Pleno01,Pleob01,Plesc01,Pratr01,Prepr01,Psabi01,Psahe01,Psaob01,Pseal01,Psebr01,Psepa01,Pseso01,Pulob01,Punra01,Reisi01,Reiun01,Rhoab01,Selat01,Selja01,Selni01,Selpu01,Selrh01,Selsa02,Stakr01,Staov01,Stapa01,Stapi02,Stasm01,Stath01,Steha01,Stein01,Stepa01,Stete01,Stetr01,Suran01,Surbr01,Surla01,Surmi01,Surro01,Sursu02,Synac01,Synac02,Synbi02,Synde01,Synul01,Tabfa01,Tabfl01,Tryan02,Tryap01,Tryco02,Tryde01,Tryku01,Tryle01,Ulnul01,Vibtr01,HERlvl1Code,IBD,IBD_EQR,Uncommon_Taxons,QuasiConstant_Numeric
SamplingOperations_code,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
S02000008_20170703,405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.407407,NaN,NaN,NaN,NaN,88.888889,NaN,NaN,NaN,NaN,2.469136,NaN,NaN,NaN,NaN,NaN,NaN,4.938272,NaN,NaN,NaN,NaN,NaN,187.654321,NaN,NaN,NaN,NaN,NaN,2.469136,NaN,NaN,NaN,44.444444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.938272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [13]:
# X is where IBD is not null
X_train = df[df['IBD'].notnull()].drop(columns=['IBD_EQR'])
y_train = X_train.pop('IBD')

X_test = df[df['IBD'].isnull()].drop(columns=['IBD_EQR'])
y_test = X_test.pop('IBD')


In [15]:
# ==== SINGLE-FIT ELASTIC NET ==================================================
# Assumes X_train, y_train, X_test are already defined DataFrames/Series.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet

RANDOM_STATE = 42
OUTDIR = "model_outputs_onefit"

# Column typing
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.columns.difference(num_cols).tolist()

# Preprocessor
num_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
    ("sc",  StandardScaler(with_mean=False))
])
cat_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])
pre = ColumnTransformer(
    transformers=[("num", num_pipe, num_cols),
                  ("cat", cat_pipe, cat_cols)],
    remainder="drop",
    sparse_threshold=1.0
)

# Model with fixed hyperparameters (set your own)
lin = ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=10000,
                 random_state=RANDOM_STATE, selection="cyclic")

pipe = Pipeline(steps=[("pre", pre), ("lin", lin)])

# Fit once on all training data
pipe.fit(X_train, y_train)

# Predict once on test
y_test_pred = pipe.predict(X_test)
pd.Series(y_test_pred, index=X_test.index, name="prediction")\
  .to_csv(f"test_predictions.csv", index=True)

print("Done. One fit, one predict. Saved to", f"test_predictions.csv")


c:\Users\herie\miniconda3\envs\a\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['Achno01' 'Actde01' 'Adlaq01' 'Birbi01' 'Calla01' 'Cenre01' 'Cochu02'
 'Cycex01' 'Cyman04' 'Cymel02' 'Cympr01' 'Diavi01' 'Dipbo02' 'Enchu01'
 'Encla01' 'Eucau01' 'Eungi01' 'Frape02' 'Gomca03' 'Hipps01' 'Meldi02'
 'Melly01' 'Melny01' 'Mesme01' 'Navel01' 'Navge03' 'Navgr02' 'Navha08'
 'Navin02' 'Navme02' 'Navmo10' 'Navpa03' 'Navpe02' 'Navra03' 'Navre01'
 'Navre06' 'Navsc05' 'Navsu17' 'Navsy02' 'Navto01' 'Navum02' 'Nitla04'
 'Nitmi04' 'Nitpl01' 'Nitta01' 'Nitzu01' 'Nuppa01' 'Pinin03' 'Pinsu08'
 'Plala01' 'Plami01' 'Psebo01' 'Stapo01' 'Stapu01' 'Surol01']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\herie\miniconda3\envs\a\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['Achno01' 'Actde01' 'Adlaq01' 'Birbi01' 'Calla01' 'Cenre01' 'C

Done. One fit, one predict. Saved to test_predictions.csv


In [16]:
y_test_pred

array([13.84136548, 16.37459627, 13.65012094, ..., 21.2228001 ,
       21.01052668, 20.81605133], shape=(5663,))

In [18]:
X_test

# add the predictions to X_test
X_test = X_test.copy()
X_test['IBD_Prediction'] = y_test_pred
X_test

TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000010_20080811                                    400      NaN      NaN   
S02000010_20100719                                    404      NaN      NaN   
S02000010_20150811                                    400      NaN      NaN   
S02000010_20160825                                    397      NaN      NaN   
S02000010_20170703                                    410      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000010_20080811           NaN        NaN      NaN      NaN      NaN   
S02000010_20100719           NaN        NaN      NaN      NaN      NaN   
S02000010_20150811           NaN        NaN      NaN      NaN      NaN   
S02000010_20160825           NaN        NaN      NaN      NaN      NaN   
S02000010_20170703           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000010_20080811           NaN  2.500000      NaN      NaN      NaN   
S02000010_20100719           NaN  9.900990      NaN      NaN      NaN   
S02000010_20150811           NaN       NaN      NaN      NaN      NaN   
S02000010_20160825           NaN       NaN      NaN      NaN      NaN   
S02000010_20170703           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000010_20080811           NaN       NaN      NaN      NaN      NaN   
S02000010_20100719           NaN       NaN      NaN      NaN      NaN   
S02000010_20150811           NaN       NaN      NaN      NaN      NaN   
S02000010_20160825           NaN       NaN      NaN      NaN      NaN   
S02000010_20170703           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN  2.283105      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN       NaN      NaN      NaN      NaN   

                         Achca04  Achch01  Achcl01  Achco01   Achco02  \
SamplingOperations_code                                      

In [42]:
df[['HERlvl1Name']]

,HERlvl1Name
SamplingOperations_code,
S02000008_20170703,ALSACE
S02000008_20200708,ALSACE
S02000010_20070906,ALSACE
S02000010_20090721,ALSACE
S02000010_20110723,ALSACE
...,...
S06940940_20100708,JURA-PREALPES DU NORD
S06940940_20230623,JURA-PREALPES DU NORD
S06960950_20160629,JURA-PREALPES DU NORD


In [43]:
# merge HERlvl1Name from df to X_test
X_test = X_test.merge(df[['HERlvl1Name']], left_index=True, right_index=True, how='left')
X_test


TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000010_20080811                                    400      NaN      NaN   
S02000010_20100719                                    404      NaN      NaN   
S02000010_20150811                                    400      NaN      NaN   
S02000010_20160825                                    397      NaN      NaN   
S02000010_20170703                                    410      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000010_20080811           NaN        NaN      NaN      NaN      NaN   
S02000010_20100719           NaN        NaN      NaN      NaN      NaN   
S02000010_20150811           NaN        NaN      NaN      NaN      NaN   
S02000010_20160825           NaN        NaN      NaN      NaN      NaN   
S02000010_20170703           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000010_20080811           NaN  2.500000      NaN      NaN      NaN   
S02000010_20100719           NaN  9.900990      NaN      NaN      NaN   
S02000010_20150811           NaN       NaN      NaN      NaN      NaN   
S02000010_20160825           NaN       NaN      NaN      NaN      NaN   
S02000010_20170703           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000010_20080811           NaN       NaN      NaN      NaN      NaN   
S02000010_20100719           NaN       NaN      NaN      NaN      NaN   
S02000010_20150811           NaN       NaN      NaN      NaN      NaN   
S02000010_20160825           NaN       NaN      NaN      NaN      NaN   
S02000010_20170703           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN  2.283105      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN       NaN      NaN      NaN      NaN   

                         Achca04  Achch01  Achcl01  Achco01   Achco02  \
SamplingOperations_code                                      

In [51]:
y = X_test[['HERlvl1Name', 'IBD_Prediction']]
y

,HERlvl1Name,IBD_Prediction
SamplingOperations_code,,
S02000010_20080811,ALSACE,13.841365
S02000010_20100719,ALSACE,16.374596
S02000010_20150811,ALSACE,13.650121
S02000010_20160825,ALSACE,14.954443
S02000010_20170703,ALSACE,15.640199
...,...,...
S06940940_20100708,JURA-PREALPES DU NORD,16.460970
S06940940_20230623,JURA-PREALPES DU NORD,15.555897
S06960950_20160629,JURA-PREALPES DU NORD,21.222800


In [56]:
def to_status(yhat: pd.DataFrame, ranges: pd.DataFrame) -> pd.DataFrame:
    """
    Adds the column 'IBD_EQR_Status_Predicted' to the `yhat` DataFrame by mapping the predicted IBD values 
    ('IBD_Predicted') into the appropriate bin defined by the [IBD_min, IBD_max) intervals in the `ranges` DataFrame 
    for the corresponding 'HERlvl1Name'. The topmost bin per region also includes its right endpoint.

    Parameters:
    ----------
    yhat : pd.DataFrame
        A DataFrame containing the predicted IBD values ('IBD_Predicted') and the corresponding 'HERlvl1Name'.
    ranges : pd.DataFrame
        A DataFrame containing the bin definitions for each 'HERlvl1Name', including columns:
        - 'HERlvl1Name': The region name.
        - 'IBD_EQR_Status': The status corresponding to the bin.
        - 'IBD_min': The lower bound of the bin (inclusive).
        - 'IBD_max': The upper bound of the bin (exclusive, except for the topmost bin).

    Returns:
    -------
    pd.DataFrame
        A copy of the `yhat` DataFrame with an additional column 'IBD_EQR_Status_Predicted', which contains the 
        mapped status for each prediction.

    Notes:
    -----
    - The function performs a cartesian merge between `yhat` and `ranges` based on 'HERlvl1Name'.
    - Each predicted value is matched to the bin where it falls within the [IBD_min, IBD_max) interval.
    - For the topmost bin in each region, the right endpoint (IBD_max) is included.
    - In case of ties (multiple bins matching a prediction), the first match is kept.
    """
    out = yhat.copy()
    out['__ix__'] = np.arange(len(out))

    # Copy ranges and calculate the maximum right endpoint for each region
    r = ranges[['HERlvl1Name', 'IBD_EQR_Status', 'IBD_min', 'IBD_max']].copy()
    r['__max_right__'] = r.groupby('HERlvl1Name')['IBD_max'].transform('max')

    # Cartesian merge by region, then keep the single interval that matches each prediction
    m = out.merge(r, on='HERlvl1Name', how='left')

    # Check if predictions fall within the bin intervals
    pred = m['IBD_Predicted'].astype(float)
    left_ok  = pred >= m['IBD_min']
    right_ok = (pred <  m['IBD_max']) | ((pred == m['IBD_max']) & (m['IBD_max'].eq(m['__max_right__'])))
    m = m[left_ok & right_ok]

    # In case of any ties, keep the first match; then map back to original rows
    m = m.sort_values(['__ix__', 'IBD_min', 'IBD_max']).drop_duplicates('__ix__', keep='first')
    status = m.set_index('__ix__')['IBD_EQR_Status']

    # Map the status back to the original DataFrame
    out['IBD_EQR_Status_Predicted'] = out['__ix__'].map(status)
    out = out.drop(columns='__ix__')
    return out

def get_results(yhat, ranges, output_file="IBD_EQR_Status_predictions_5663.csv") -> pd.Series:
    yhat2 = to_status(yhat, ranges)
    send_predictions = yhat2['IBD_EQR_Status_Predicted']
    send_predictions = send_predictions.to_frame()
    send_predictions = send_predictions.rename(columns={'IBD_EQR_Status_Predicted': 'IBD_EQR_Status'})
    # send_predictions.to_csv(output_file, index=True)
    print(f"Saved predictions to {output_file}")
    return send_predictions

In [28]:
ranges = pd.read_csv("ranges.csv")
ranges

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid,HERlvl1Code
0,ALPES INTERNES,Bad,0.000,9.800,9.30,2
1,ALPES INTERNES,Poor,9.800,13.225,10.30,2
2,ALPES INTERNES,Moderate,13.225,17.025,16.15,2
3,ALPES INTERNES,Good,17.025,18.725,17.90,2
4,ALPES INTERNES,High,18.725,20.000,19.55,2
...,...,...,...,...,...,...
105,VOSGES,Bad,0.000,9.550,7.90,4
106,VOSGES,Poor,9.550,12.775,11.20,4
107,VOSGES,Moderate,12.775,15.700,14.35,4
108,VOSGES,Good,15.700,18.075,17.05,4


In [57]:
# # change column name to IBD_EQR_Status_Predicted
# y = y.rename("IBD_EQR_Status_Predicted")
y

,HERlvl1Name,IBD_Prediction
SamplingOperations_code,,
S02000010_20080811,ALSACE,13.841365
S02000010_20100719,ALSACE,16.374596
S02000010_20150811,ALSACE,13.650121
S02000010_20160825,ALSACE,14.954443
S02000010_20170703,ALSACE,15.640199
...,...,...
S06940940_20100708,JURA-PREALPES DU NORD,16.460970
S06940940_20230623,JURA-PREALPES DU NORD,15.555897
S06960950_20160629,JURA-PREALPES DU NORD,21.222800


In [64]:
# renamea the column IBD_Prediction to IBD_EQR_Status_Predicted
y = y.rename(columns={'IBD_Prediction': 'IBD_EQR_Status_Predicted'})

y = y.rename(columns={'IBD_EQR_Status_Predicted': 'IBD_Predicted'})
y

,HERlvl1Name,IBD_Predicted
SamplingOperations_code,,
S02000010_20080811,ALSACE,13.841365
S02000010_20100719,ALSACE,16.374596
S02000010_20150811,ALSACE,13.650121
S02000010_20160825,ALSACE,14.954443
S02000010_20170703,ALSACE,15.640199
...,...,...
S06940940_20100708,JURA-PREALPES DU NORD,16.460970
S06940940_20230623,JURA-PREALPES DU NORD,15.555897
S06960950_20160629,JURA-PREALPES DU NORD,21.222800


In [59]:
res = get_results(y, ranges)
res

KeyError: 'IBD_Predicted'

In [ ]:
out = y.copy()
out['__ix__'] = np.arange(len(out))

# Copy ranges and calculate the maximum right endpoint for each region
r = ranges[['HERlvl1Name', 'IBD_EQR_Status', 'IBD_min', 'IBD_max']].copy()
r['__max_right__'] = r.groupby('HERlvl1Name')['IBD_max'].transform('max')

# Cartesian merge by region, then keep the single interval that matches each prediction
m = out.merge(r, on='HERlvl1Name', how='left')

# Check if predictions fall within the bin intervals
pred = m['IBD_Predicted'].astype(float)
left_ok  = pred >= m['IBD_min']
right_ok = (pred <  m['IBD_max']) | ((pred == m['IBD_max']) & (m['IBD_max'].eq(m['__max_right__'])))
m = m[left_ok & right_ok]

# In case of any ties, keep the first match; then map back to original rows
m = m.sort_values(['__ix__', 'IBD_min', 'IBD_max']).drop_duplicates('__ix__', keep='first')
status = m.set_index('__ix__')['IBD_EQR_Status']

# Map the status back to the original DataFrame
out['IBD_EQR_Status_Predicted'] = out['__ix__'].map(status)
out = out.drop(columns='__ix__')



NameError: name 'yhat2' is not defined

In [66]:

send_predictions = out['IBD_EQR_Status_Predicted']
send_predictions = send_predictions.to_frame()
send_predictions = send_predictions.rename(columns={'IBD_EQR_Status_Predicted': 'IBD_EQR_Status'})


In [ ]:
send_predictions.to_csv("Regresion_Lineal_99%.csv", index=True)

,IBD_EQR_Status
SamplingOperations_code,
S02000010_20080811,Moderate
S02000010_20100719,Good
S02000010_20150811,Moderate
S02000010_20160825,Good
S02000010_20170703,Good
...,...
S06940940_20100708,Good
S06940940_20230623,Moderate
S06960950_20160629,NaN


In [ ]:
# ==== CONFIG ==================================================================
# %pip install -U numpy pandas scikit-learn matplotlib scipy

RANDOM_STATE = 42
N_JOBS = -1
PRIMARY_SCORING = "neg_mean_absolute_error"  # alt: "neg_root_mean_squared_error", "r2"
OUTDIR = "model_outputs"

# If your data is in CSVs, load here; else set X_train, y_train, X_test directly.
# import pandas as pd
# X_train = pd.read_csv("X_train.csv")
# y_train = pd.read_csv("y_train.csv").squeeze("columns")
# X_test  = pd.read_csv("X_unlabeled_test.csv")

# ==== LIBRARIES ===============================================================
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import (
    GridSearchCV, RepeatedKFold, KFold, cross_val_predict, cross_validate, learning_curve
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==== DATA CHECKS =============================================================
assert isinstance(X_train, pd.DataFrame)
assert isinstance(y_train, (pd.Series, np.ndarray))
assert len(X_train) == len(y_train)
assert isinstance(X_test, pd.DataFrame)
os.makedirs(OUTDIR, exist_ok=True)

# ==== COLUMN TYPING ===========================================================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.columns.difference(num_cols).tolist()

# ==== PREPROCESSOR ============================================================
num_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
    ("sc",  StandardScaler(with_mean=False))
])

cat_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

pre = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# ==== MODEL + HYPERPARAMS =====================================================
lin = ElasticNet(max_iter=10000, random_state=RANDOM_STATE, selection="cyclic")
pipe = Pipeline(steps=[("pre", pre), ("lin", lin)])

param_grid = {
    "lin__alpha":    np.logspace(-4, 2, 25),
    "lin__l1_ratio": [0.05, 0.2, 0.95]
}

inner_cv = 4
search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring=PRIMARY_SCORING,
    cv=inner_cv,
    n_jobs=N_JOBS,
    refit=True,
    verbose=0
)

# ==== OOF PREDICTIONS (partitioned CV) ========================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

y_oof = cross_val_predict(
    estimator=search,
    X=X_train, y=y_train,
    cv=outer_cv,
    n_jobs=N_JOBS,
    method="predict",
    verbose=0
)

# (Optional) repeated-CV score for robustness (not used for OOF)
rep_cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
cv_res = cross_validate(
    search, X_train, y_train, cv=rep_cv,
    scoring=PRIMARY_SCORING, n_jobs=N_JOBS, return_estimator=False
)
print(f"RepeatedCV mean {PRIMARY_SCORING}: {np.mean(cv_res['test_score']):.6f}")

# ==== METRICS =================================================================
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mdae = np.median(np.abs(y_true - y_pred))
    eps = np.finfo(float).eps
    mask = np.abs(y_true) > eps
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
    rho, _ = stats.spearmanr(y_true, y_pred)
    return dict(MAE=mae, RMSE=rmse, R2=r2, MedAE=mdae, MAPE_pct=mape, Spearman=rho)

metrics_oof = regression_metrics(y_train, y_oof)
print("=== Cross-validated metrics on train (OOF) ===")
for k, v in metrics_oof.items():
    print(f"{k:>10}: {v:.6f}")

pd.DataFrame({"y_true": y_train, "y_oof": y_oof}).to_csv(f"{OUTDIR}/oof_predictions.csv", index=False)

# ==== CONFORMAL INTERVALS FROM OOF RESIDUALS =================================
resid = np.asarray(y_train) - np.asarray(y_oof)

def conformal_q(residuals, alpha):
    return np.quantile(np.abs(residuals), 1 - alpha, method="higher")

q90 = conformal_q(resid, 0.10)
q95 = conformal_q(resid, 0.05)
print(f"\nConformal |residual| quantiles: q90={q90:.6f}, q95={q95:.6f}")

# ==== FINAL FIT ON ALL TRAIN ==================================================
search.fit(X_train, y_train)
best_pipe = search.best_estimator_
print("\nBest hyperparameters:", search.best_params_)

# ==== PREDICT TEST + PIs ======================================================
y_test_pred = best_pipe.predict(X_test)
test_df = pd.DataFrame({
    "prediction": y_test_pred,
    "pi90_lo": y_test_pred - q90,
    "pi90_hi": y_test_pred + q90,
    "pi95_lo": y_test_pred - q95,
    "pi95_hi": y_test_pred + q95
}, index=X_test.index)
test_path = f"{OUTDIR}/test_predictions_with_PIs.csv"
test_df.to_csv(test_path)
print(f"\nSaved test predictions with 90%/95% conformal PIs → {test_path}")

# ==== FEATURE IMPORTANCE (TOP COEFFICIENTS) ===================================
def get_feature_names(preprocessor: ColumnTransformer) -> np.ndarray:
    names = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if hasattr(trans, "get_feature_names_out"):
            base_names = cols if isinstance(cols, list) else preprocessor.feature_names_in_[cols]
            gn = trans.get_feature_names_out(base_names)
            names.extend(gn)
        else:
            names.extend(list(cols if isinstance(cols, list) else preprocessor.feature_names_in_[cols]))
    return np.array(names, dtype=object)

coef = np.asarray(best_pipe.named_steps["lin"].coef_).ravel()
feat_names = get_feature_names(best_pipe.named_steps["pre"])
top_k = min(20, (coef != 0).sum() if (coef != 0).any() else 20)
idx = np.argsort(np.abs(coef))[-top_k:][::-1]
top_features = pd.DataFrame({
    "feature": feat_names[idx],
    "coef": coef[idx],
    "abs_coef": np.abs(coef[idx])
})
top_features.to_csv(f"{OUTDIR}/top_coefficients.csv", index=False)
print(f"Saved top coefficients → {OUTDIR}/top_coefficients.csv")

# ==== PLOTS ===================================================================
plt.figure(figsize=(6,6))
lim_lo = np.nanpercentile(np.concatenate([np.asarray(y_train), np.asarray(y_oof)]), 1)
lim_hi = np.nanpercentile(np.concatenate([np.asarray(y_train), np.asarray(y_oof)]), 99)
plt.scatter(y_train, y_oof, s=6, alpha=0.4)
plt.plot([lim_lo, lim_hi], [lim_lo, lim_hi], lw=2)
plt.xlabel("Actual (train)")
plt.ylabel("OOF prediction")
plt.title("Parity plot (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_parity_oof.png", dpi=160)

plt.figure(figsize=(7,4))
plt.scatter(y_oof, resid, s=6, alpha=0.4)
plt.axhline(0, lw=1)
plt.xlabel("OOF prediction")
plt.ylabel("Residual")
plt.title("Residuals vs fitted (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_vs_fit.png", dpi=160)

plt.figure(figsize=(7,4))
plt.hist(resid, bins=60, alpha=0.8)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Residual histogram (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_hist.png", dpi=160)

plt.figure(figsize=(6,6))
stats.probplot(resid, dist="norm", plot=plt)
plt.title("QQ plot of residuals (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_qq.png", dpi=160)

train_sizes, train_scores, valid_scores = learning_curve(
    estimator=best_pipe,
    X=X_train, y=y_train,
    cv=5,
    scoring=PRIMARY_SCORING,
    n_jobs=N_JOBS,
    train_sizes=np.linspace(0.1, 1.0, 6),
    shuffle=True,
    random_state=RANDOM_STATE
)
plt.figure(figsize=(7,4))
plt.plot(train_sizes, -train_scores.mean(axis=1), marker="o", label="Train MAE")
plt.plot(train_sizes, -valid_scores.mean(axis=1), marker="o", label="CV MAE")
plt.xlabel("Training examples")
plt.ylabel("MAE")
plt.title("Learning curve (Elastic Net)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_learning_curve.png", dpi=160)

plt.figure(figsize=(8,6))
plt.barh(top_features["feature"][::-1], top_features["coef"][::-1])
plt.xlabel("Coefficient")
plt.title("Top coefficients (Elastic Net)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_top_coefs.png", dpi=160)

print("\nPlots saved in:", OUTDIR)


: 